In [ ]:
import pandas as pd
from pathlib import Path
from ml.feature_engineering import add_lag_features, add_temporal_columns
from ml.preprocessing import time_series_split, select_feature_columns, build_preprocessor
from ml.feature_spec import TARGET_COLUMN

In [ ]:
raw_df = pd.read_csv('uganda_drug_supply_synthetic.csv')
raw_df.head()

In [ ]:
df = add_temporal_columns(raw_df)
df = add_lag_features(df)
train_df, holdout_df = time_series_split(df, test_size=0.2)
val_df, test_df = time_series_split(holdout_df, test_size=0.5)
print('train/val/test:', len(train_df), len(val_df), len(test_df))

In [ ]:
feature_cols = select_feature_columns(train_df)
preprocessor = build_preprocessor(feature_cols)
X_train = preprocessor.fit_transform(train_df[feature_cols])
X_val = preprocessor.transform(val_df[feature_cols])
X_test = preprocessor.transform(test_df[feature_cols])
y_train = train_df[TARGET_COLUMN].copy()
y_val = val_df[TARGET_COLUMN].copy()
y_test = test_df[TARGET_COLUMN].copy()
print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
assert 'lag_1' in train_df.columns
assert 'rolling_mean_3' in train_df.columns
assert train_df['lag_1'].isna().sum() > 0
train_df[['average_monthly_demand', 'lag_1', 'lag_2', 'rolling_mean_3', 'rolling_std_3', 'demand_growth_rate']].head(10)

In [ ]:
import joblib
from config import ARTIFACTS_DIR
artifact = ARTIFACTS_DIR / 'preprocessing.joblib'
joblib.dump({'preprocessor': preprocessor, 'feature_columns': feature_cols}, artifact)
print('saved', artifact)

In [ ]:
from config import PROCESSED_DATA_DIR
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
train_df.to_parquet(PROCESSED_DATA_DIR / 'train.parquet', index=False)
val_df.to_parquet(PROCESSED_DATA_DIR / 'val.parquet', index=False)
test_df.to_parquet(PROCESSED_DATA_DIR / 'test.parquet', index=False)
data_dict = pd.DataFrame({'column': df.columns, 'dtype': df.dtypes.astype(str)})
data_dict.to_csv(PROCESSED_DATA_DIR / 'data_dictionary.csv', index=False)
print('processed datasets + dictionary exported')